# ML 데이터마트

## 변경사항 

**핵심 변경: 품질 점수 percentile rank를 train fit -> val/test transform 방식으로 변경**

v4에서는 split을 먼저 하고 라벨 threshold(상위 25% 등)를 train 기준으로 잡았으나,
**품질 점수 자체는 여전히 전체 valid 데이터로 percentile rank를 계산**하고 있었음.
이 경우 val/test 행의 분포 정보가 train 행의 점수 계산에 섞여 들어가
**약한 형태의 target leakage**가 발생함.

v5에서는 다음과 같이 수정:

1. **GroupedECDFScorer 도입**: train 기준 그룹별 ECDF(경험적 분포)를 fit한 뒤,
   val/test는 해당 분포에 끼워 넣어 percentile을 부여
2. **Fallback 계층화**: train 그룹 표본 < 30이면 전역 train 분포 사용,
   < 10이면 0.5(중립) 부여. 각 행에 fallback level 기록
3. **ctit_median**: 전체 valid -> df_train_raw 기준으로 변경
4. **GLOBAL_CVR**: 전체 valid -> df_train_raw 기준으로 변경
   (smoothed_cvr 계산이 split 이후로 이동)
5. **Stratify 키 변경**: 점수 quintile -> log_margin quintile (raw 변수 기반)
   품질 점수가 split 이후에 계산되므로 stratify에 사용할 수 없음. 가중치 비중이
   가장 큰 raw 변수(log_margin, 0.35)를 대리 키로 사용
6. **순서 재배치**: split을 raw 변수 가공 직후로 이동 -> 이후 모든 fit은 train에서

## 누수 차단 흐름

```
[1] valid 생성 (is_valid_click10==1 필터)
[2] raw 변수 가공: total_margin, complete_cnt_f, avg_ctit, log_*
    -- ctit_median, log_*는 변환만 적용 (fit 없음)
[3] stratify 키 생성: log_margin quintile
[4] Split (train/val/test)
[5] Train fit:
      - GLOBAL_CVR (train)
      - ctit_median (train)
      - GroupedECDFScorer (train)
[6] 모든 split에 transform 적용
      - smoothed_cvr (train GLOBAL_CVR 사용)
      - avg_ctit_imputed (train ctit_median 사용)
      - score_* (train ECDF 사용)
      - quality_score_LABEL_ONLY (가중합)
[7] 라벨 생성:
      - 모델1 threshold (train 상위 25%)
      - 모델2 group thresholds (train 그룹별 하위 25%)
[8] 이후 단계는 v4와 동일
```

## 보고서 추가 명시 사항

- v4 -> v5: target leakage 약한 형태 차단
- Fallback 사용 비율: val/test 별, 변수 별로 출력
- Stratify 키 변경: 점수 quintile -> log_margin quintile


## 0. 환경 설정

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, joblib
from scipy.stats import rankdata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE = r'C:\Users\JMJEON\눌러조\streamlit\data' + '\\'
OUT  = r'C:\Users\JMJEON\눌러조\streamlit\models' + '\\'
SEED = 42

files = ['ad_outcome.parquet','ad_attr_map.parquet',
         'finance_clean1.parquet','finance_clean2.parquet',
         'sched_clean.parquet','ad_master_clean.parquet',
         'ive_ad_classification.parquet']
for f in files:
    print(f"{'OK' if os.path.exists(BASE+f) else 'MISSING'} {f}")

OK ad_outcome.parquet
OK ad_attr_map.parquet
OK finance_clean1.parquet
OK finance_clean2.parquet
OK sched_clean.parquet
OK ad_master_clean.parquet
OK ive_ad_classification.parquet


## 1. 데이터 로드

In [2]:
print("로드 중...")
ad_outcome = pd.read_parquet(BASE + 'ad_outcome.parquet')
ad_attr    = pd.read_parquet(BASE + 'ad_attr_map.parquet')
finance1   = pd.read_parquet(BASE + 'finance_clean1.parquet')
finance2   = pd.read_parquet(BASE + 'finance_clean2.parquet')
sched      = pd.read_parquet(BASE + 'sched_clean.parquet')
ad_master  = pd.read_parquet(BASE + 'ad_master_clean.parquet')
ad_class   = pd.read_parquet(BASE + 'ive_ad_classification.parquet')

finance = pd.concat([finance1, finance2], ignore_index=True)
print(f"ad_outcome : {ad_outcome.shape}")
print(f"ad_attr    : {ad_attr.shape}")
print(f"finance    : {finance.shape}")
print(f"sched      : {sched.shape}")
print(f"ad_master  : {ad_master.shape}")
print(f"ad_class   : {ad_class.shape}")

로드 중...


ad_outcome : (445260, 22)
ad_attr    : (445260, 18)
finance    : (2858560, 33)
sched      : (473066, 21)
ad_master  : (445260, 73)
ad_class   : (445260, 12)


## 2. 유효 광고 선정 + 이상치 Flag

### click_cnt >= 30 필터 적용 (v5 변경: 기존 10 → 30)
click_cnt >= 30: 클릭 >= 30건 광고만 선정.
이 모델의 적용 대상은 최소 30건 이상 클릭이 확보된 광고로 제한된다.

### is_outlier (v4 유지)
최종 성과 기반 값이므로 피처 투입 금지. row filtering 메타 컬럼으로만 저장.

In [3]:
valid = ad_outcome[ad_outcome['click_cnt'] >= 30].copy()
total = len(ad_outcome)
print(f"전체: {total:,} -> 유효(클릭>=30): {len(valid):,}건 ({len(valid)/total*100:.1f}%)")
print("주의: 모델 적용 대상은 클릭 >= 30 확보 광고로 제한됨 (표본 선택 편향 가능성 보고서 명시 필요)")

fin_agg = finance.groupby('ads_idx').agg(
    total_margin   = ('ive_margin', 'sum'),
    complete_cnt_f = ('rwd_idx', 'count'),
    avg_ctit_fin   = ('ctit', 'mean'),
).reset_index()
assert fin_agg['ads_idx'].nunique() == len(fin_agg)

valid = valid.merge(fin_agg, on='ads_idx', how='left')
valid = valid.drop(columns=['avg_ctit'], errors='ignore')
valid = valid.rename(columns={'avg_ctit_fin': 'avg_ctit'})

CLICK_P99 = valid['click_cnt'].quantile(0.99)
valid['is_outlier_cvr']    = (valid['cvr_pct'] >= 99.9).astype(int)
valid['is_outlier_click']  = (valid['click_cnt'] > CLICK_P99).astype(int)
valid['is_outlier_margin'] = (valid['total_margin'] < 0).astype(int)
valid['is_outlier'] = (
    (valid['is_outlier_cvr'] == 1) |
    (valid['is_outlier_click'] == 1) |
    (valid['is_outlier_margin'] == 1)
).astype(int)

print(f"\n이상치: {valid['is_outlier'].sum():,}건 ({valid['is_outlier'].mean()*100:.1f}%) — 피처 투입 금지")
assert valid['click_cnt'].min() >= 30
print("OK: 검증 통과")

전체: 445,260 -> 유효(클릭>=30): 3,628건 (0.8%)
주의: 모델 적용 대상은 클릭 >= 30 확보 광고로 제한됨 (표본 선택 편향 가능성 보고서 명시 필요)

이상치: 120건 (3.3%) — 피처 투입 금지
OK: 검증 통과


## 3. Raw 변수 가공 (점수 계산 전)

### v5 변경: 점수 계산을 split 이후로 이동
v4에서는 이 단계에서 percentile rank까지 계산했으나,
v5에서는 split 이후 train 기준으로만 계산하기 위해
**여기서는 raw 변수 가공만 수행**한다.

- 결측 채움 (total_margin, complete_cnt_f -> 0)
- log 변환 (단조 변환이라 fit 불필요)
- avg_ctit는 결측 그대로 둔다 (median은 split 이후 train에서 fit)
- is_ctit_null flag 생성

### 주의
- ctit_median, GLOBAL_CVR, smoothed_cvr, score_*, quality_score는 **여기서 만들지 않음**
- 이후 split을 거친 뒤 train fit 객체로 transform

In [4]:
valid['total_margin']   = valid['total_margin'].fillna(0)
valid['complete_cnt_f'] = valid['complete_cnt_f'].fillna(0)

# log 변환은 단조함수라 fit 불필요 (train/val/test 행마다 독립적으로 적용 가능)
valid['log_margin']   = np.log1p(valid['total_margin'].clip(lower=0))
valid['log_complete'] = np.log1p(valid['complete_cnt_f'])

# avg_ctit 결측 flag만 생성 (median 대체는 split 이후)
valid['is_ctit_null'] = valid['avg_ctit'].isna().astype(int)
print(f"avg_ctit null: {valid['avg_ctit'].isna().sum():,}건 (median 대체는 split 이후 train 기준으로 적용)")

print("OK: raw 변수 가공 완료 (점수 계산은 split 이후)")

avg_ctit null: 89건 (median 대체는 split 이후 train 기준으로 적용)
OK: raw 변수 가공 완료 (점수 계산은 split 이후)


## 4. 초기 분할 — Split 먼저

### 변경사항: stratify 키를 raw 변수 기반으로 변경
v4에서는 quality_score_LABEL_ONLY의 quintile로 stratify했으나,
v5에서는 점수 자체가 split 이후 계산되므로 사용 불가.

대안: 가중합 점수에서 비중이 가장 큰 raw 변수(log_margin, 0.35)의 quintile로 stratify.
점수가 raw 변수들의 가중합이므로 분포 균형 측면에서 유사한 효과를 기대할 수 있다.

### 변경 효과
- split 결과는 v4와 거의 유사 (점수 = 4개 변수의 가중합이라 log_margin과 강한 상관)
- 보고서에 "stratify 키 변경 (점수 quintile -> log_margin quintile)" 명시

In [5]:
# stratify 키: log_margin quintile (가중치 0.35로 가장 큰 변수)
valid['_strat'] = pd.qcut(
    valid['log_margin'], q=5, labels=False, duplicates='drop'
)

idx_all = valid.index.tolist()
idx_train, idx_temp = train_test_split(
    idx_all, test_size=0.4, random_state=SEED,
    stratify=valid.loc[idx_all, '_strat']
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.5, random_state=SEED,
    stratify=valid.loc[idx_temp, '_strat']
)

df_train_raw = valid.loc[idx_train].copy()
df_val_raw   = valid.loc[idx_val].copy()
df_test_raw  = valid.loc[idx_test].copy()

print(f"Train: {len(df_train_raw):,}건 ({len(df_train_raw)/len(valid)*100:.0f}%)")
print(f"Val:   {len(df_val_raw):,}건 ({len(df_val_raw)/len(valid)*100:.0f}%)")
print(f"Test:  {len(df_test_raw):,}건 ({len(df_test_raw)/len(valid)*100:.0f}%)")

# 유형 분포 확인 (split이 유형별로 균등한지)
print("\n[유형별 train/val/test 비율 — 큰 편차 없는지 확인]")
for t in valid['analysis_ads_type_label'].unique():
    n_tr = (df_train_raw['analysis_ads_type_label']==t).sum()
    n_v  = (df_val_raw['analysis_ads_type_label']==t).sum()
    n_te = (df_test_raw['analysis_ads_type_label']==t).sum()
    print(f"  {t}: train={n_tr}, val={n_v}, test={n_te}")

Train: 2,176건 (60%)
Val:   726건 (20%)
Test:  726건 (20%)

[유형별 train/val/test 비율 — 큰 편차 없는지 확인]
  참여형: train=2042, val=683, test=694
  설치형: train=33, val=11, test=8
  네이버: train=2, val=0, test=0
  퀘스트: train=53, val=14, test=11
  CPS(물건구매): train=21, val=7, test=4
  유튜브: train=1, val=0, test=1
  실행형: train=21, val=9, test=5
  클릭형: train=2, val=1, test=0
  제휴: train=0, val=0, test=1
  인스타: train=1, val=1, test=2


## 5. Train Fit (1) — ctit_median, GLOBAL_CVR

### v5 변경: train 기준으로 변경
- v4: `ctit_median = valid['avg_ctit'].median()` (전체)
- v5: `ctit_median = df_train_raw['avg_ctit'].median()` (train만)

같은 방식으로 GLOBAL_CVR도 train 기준 재계산.

### 적용 범위
- ctit_median: 모든 split의 avg_ctit 결측 대체에 사용
- GLOBAL_CVR: smoothed_cvr 계산의 prior로 사용 (라벨 생성 전용)

In [6]:
# ctit_median: train fit
ctit_median = df_train_raw['avg_ctit'].median()
print(f"[Train fit] ctit_median = {ctit_median:.0f}초")

# GLOBAL_CVR: train fit (smoothed_cvr 의 prior)
GLOBAL_CVR = (df_train_raw['complete_cnt'].sum() /
              df_train_raw['click_cnt'].sum())
print(f"[Train fit] GLOBAL_CVR = {GLOBAL_CVR*100:.2f}%")

# 모든 split에 transform 적용
def apply_train_fits(df):
    df = df.copy()
    df['avg_ctit_imputed'] = df['avg_ctit'].fillna(ctit_median)
    df['log_ctit']         = np.log1p(df['avg_ctit_imputed'].clip(lower=0))
    df['smoothed_cvr']     = (
        (df['complete_cnt'] + 10 * GLOBAL_CVR) /
        (df['click_cnt']    + 10) * 100
    )
    return df

df_train_raw = apply_train_fits(df_train_raw)
df_val_raw   = apply_train_fits(df_val_raw)
df_test_raw  = apply_train_fits(df_test_raw)

# 검증
for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
    assert df['smoothed_cvr'].between(0, 100).all(), f"{name} smoothed_cvr 범위 이탈"
    assert df['smoothed_cvr'].isna().sum() == 0, f"{name} smoothed_cvr 결측"
    assert df['avg_ctit_imputed'].isna().sum() == 0, f"{name} avg_ctit_imputed 결측"

print("\nOK: 모든 split에 train fit 변환 완료")

[Train fit] ctit_median = 56초
[Train fit] GLOBAL_CVR = 47.52%

OK: 모든 split에 train fit 변환 완료


## 6. Train Fit (2) — GroupedECDFScorer 정의

### v5 핵심 추가: 그룹별 ECDF 기반 점수 변환기

**fit**: train 데이터에서 (group, value_col) 별로 정렬된 값 배열을 저장
**transform**: 새 데이터 행은 train 정렬 배열에서의 mid-rank percentile을 부여

### Fallback 계층
- 그룹 표본 >= 30 (`min_group_size`): 그룹 ECDF 사용 (level 0)
- 그룹 표본 < 30 + 전역 표본 >= 10 (`fallback_min_size`): 전역 ECDF 사용 (level 1)
- 둘 다 미달: 0.5 (level 2)

### 누수 차단의 핵심
- val/test 행은 **자기 자신의 분포에 영향을 미치지 않는다**
- 모든 percentile은 train 분포에 대한 상대 위치

In [7]:
class GroupedECDFScorer:
    """
    train fit -> val/test/production transform 가능한 그룹별 ECDF 점수 변환기.

    train의 분포에 val/test 값을 끼워 넣어 percentile을 부여한다.
    val/test 행은 자기 자신의 분포에 영향을 미치지 않으므로 누수가 발생하지 않는다.

    Parameters
    ----------
    group_col : str
        그룹핑 컬럼명 (예: 'analysis_ads_type_label')
    value_cols : list of str
        percentile rank를 계산할 컬럼들
    invert_cols : list of str, optional
        '낮을수록 좋은' 변수. transform 시 1 - p 로 반전.
    min_group_size : int
        그룹별 최소 표본 수. 미달 시 전역 fallback.
    fallback_min_size : int
        전역 fallback 최소 표본 수. 미달 시 0.5.
    """
    def __init__(self, group_col, value_cols, invert_cols=None,
                 min_group_size=30, fallback_min_size=10):
        self.group_col = group_col
        self.value_cols = list(value_cols)
        self.invert_cols = set(invert_cols or [])
        self.min_group_size = min_group_size
        self.fallback_min_size = fallback_min_size
        self.ecdf_ = {}             # {(group, col): sorted np.array}
        self.global_ecdf_ = {}      # {col: sorted np.array}
        self.group_sizes_ = {}      # {group: n}
        self.fitted_ = False

    def fit(self, df):
        # 전역 ECDF (모든 train 데이터)
        for col in self.value_cols:
            arr = df[col].dropna().values
            self.global_ecdf_[col] = np.sort(arr)

        # 그룹별 ECDF (min_group_size 이상인 그룹만)
        for g, sub in df.groupby(self.group_col):
            self.group_sizes_[g] = len(sub)
            if len(sub) >= self.min_group_size:
                for col in self.value_cols:
                    arr = sub[col].dropna().values
                    if len(arr) >= self.min_group_size:
                        self.ecdf_[(g, col)] = np.sort(arr)
        self.fitted_ = True
        return self

    @staticmethod
    def _percentile_in(sorted_vals, x):
        """sorted 배열에서 x의 mid-rank percentile (0~1)."""
        n = len(sorted_vals)
        if n == 0 or n == 1:
            return 0.5
        left  = np.searchsorted(sorted_vals, x, side='left')
        right = np.searchsorted(sorted_vals, x, side='right')
        avg_rank = (left + right) / 2
        # (rank - 0.5) / n 의 mid-rank 버전
        return np.clip((avg_rank + 0.5) / n, 0.0, 1.0)

    def transform(self, df):
        assert self.fitted_, "fit() 먼저 호출"
        n = len(df)
        out = pd.DataFrame(index=df.index)
        groups = df[self.group_col].values

        for col in self.value_cols:
            scores = np.full(n, 0.5)
            levels = np.zeros(n, dtype=int)  # 0=group, 1=global, 2=missing
            vals = df[col].values

            # 그룹 단위로 벡터화
            unique_groups = pd.unique(groups)
            for g in unique_groups:
                mask = groups == g
                sub_vals = vals[mask]
                if (g, col) in self.ecdf_:
                    sorted_train = self.ecdf_[(g, col)]
                    lvl = 0
                elif len(self.global_ecdf_.get(col, [])) >= self.fallback_min_size:
                    sorted_train = self.global_ecdf_[col]
                    lvl = 1
                else:
                    scores[mask] = 0.5
                    levels[mask] = 2
                    continue

                # vectorized searchsorted
                valid_mask = ~pd.isna(sub_vals)
                sub_scores = np.full(mask.sum(), 0.5)
                sub_levels = np.full(mask.sum(), lvl, dtype=int)
                if valid_mask.any():
                    v = sub_vals[valid_mask].astype(float)
                    n_train = len(sorted_train)
                    if n_train > 1:
                        left  = np.searchsorted(sorted_train, v, side='left')
                        right = np.searchsorted(sorted_train, v, side='right')
                        avg_rank = (left + right) / 2
                        p = np.clip((avg_rank + 0.5) / n_train, 0.0, 1.0)
                    else:
                        p = np.full(len(v), 0.5)
                    if col in self.invert_cols:
                        p = 1 - p
                    sub_scores[valid_mask] = p
                # 결측은 0.5, level 2
                sub_levels[~valid_mask] = 2
                sub_scores[~valid_mask] = 0.5

                scores[mask] = sub_scores
                levels[mask] = sub_levels

            out[f'score_{col}'] = scores
            out[f'fallback_{col}'] = levels

        return out


# ECDF Scorer fit on train
SCORE_VALUE_COLS = ['smoothed_cvr', 'log_margin', 'log_complete', 'log_ctit']
scorer = GroupedECDFScorer(
    group_col='analysis_ads_type_label',
    value_cols=SCORE_VALUE_COLS,
    invert_cols=['log_ctit'],   # ctit는 낮을수록 좋음
    min_group_size=30,
    fallback_min_size=10,
).fit(df_train_raw)

print(f"[Train fit] GroupedECDFScorer")
print(f"  대상 변수: {SCORE_VALUE_COLS}")
print(f"  반전 변수: {sorted(scorer.invert_cols)}")
print(f"  그룹 ECDF 등록: {len(scorer.ecdf_) // len(SCORE_VALUE_COLS)}개 그룹")
print(f"  전체 그룹 수: {len(scorer.group_sizes_)}개")
print(f"  fallback 대상 그룹(n<30): "
      f"{sum(1 for n in scorer.group_sizes_.values() if n < 30)}개")

[Train fit] GroupedECDFScorer
  대상 변수: ['smoothed_cvr', 'log_margin', 'log_complete', 'log_ctit']
  반전 변수: ['log_ctit']
  그룹 ECDF 등록: 3개 그룹
  전체 그룹 수: 9개
  fallback 대상 그룹(n<30): 6개


## 7. Transform — 점수 계산 + 가중합

### 각 split에 train ECDF로 transform 적용
- val/test 행은 train 분포 위에 끼워 넣어진 percentile만 부여 받음
- 점수 컬럼 + fallback level 컬럼 동시 생성

### Fallback 비율 보고
val/test에서 fallback level 1, 2 사용 비율을 출력하여 보고서에 명시.

In [8]:
# 컬럼명 매핑: scorer가 만드는 'score_log_margin' -> 기존 'score_margin' 등으로 통일
SCORE_COL_MAP = {
    'score_smoothed_cvr': 'score_cvr',
    'score_log_margin':   'score_margin',
    'score_log_complete': 'score_complete',
    'score_log_ctit':     'score_ctit',
}
FALLBACK_COL_MAP = {
    'fallback_smoothed_cvr': 'fb_cvr',
    'fallback_log_margin':   'fb_margin',
    'fallback_log_complete': 'fb_complete',
    'fallback_log_ctit':     'fb_ctit',
}

def apply_scorer(df):
    df = df.copy()
    score_df = scorer.transform(df)
    score_df = score_df.rename(columns={**SCORE_COL_MAP, **FALLBACK_COL_MAP})
    for c in score_df.columns:
        df[c] = score_df[c].values
    # 가중합 -> quality_score_LABEL_ONLY
    df['quality_score_LABEL_ONLY'] = (
        df['score_margin']   * 0.35 +
        df['score_cvr']      * 0.30 +
        df['score_complete'] * 0.20 +
        df['score_ctit']     * 0.15
    ) * 100
    return df

df_train_raw = apply_scorer(df_train_raw)
df_val_raw   = apply_scorer(df_val_raw)
df_test_raw  = apply_scorer(df_test_raw)

# 검증
for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
    assert df['quality_score_LABEL_ONLY'].between(0, 100).all(), f"{name} 점수 범위 이탈"
    assert df['quality_score_LABEL_ONLY'].isna().sum() == 0, f"{name} 점수 결측"

# 유형별 평균 점수 확인 (train은 정의상 ~50 수렴, val/test는 fallback 영향 받음)
print("[유형별 평균 점수 — train ~50, val/test 편차 확인]")
for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
    avg = df.groupby('analysis_ads_type_label')['quality_score_LABEL_ONLY'].mean().round(1)
    out_of_range = avg[(avg < 40) | (avg > 60)]
    print(f"  {name}: 평균 범위 [{avg.min():.1f}, {avg.max():.1f}], "
          f"40~60 이탈 유형={len(out_of_range)}개")

# Fallback 비율 보고
print("\n[Fallback 사용 비율 — 보고서 명시]")
for fb_col in ['fb_cvr', 'fb_margin', 'fb_complete', 'fb_ctit']:
    print(f"  {fb_col}:")
    for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
        lvl = df[fb_col]
        pct_global = (lvl == 1).mean() * 100
        pct_miss   = (lvl == 2).mean() * 100
        print(f"    {name}: global_fallback={pct_global:.2f}%, missing={pct_miss:.2f}%")

print("\nOK: 점수 계산 완료")

[유형별 평균 점수 — train ~50, val/test 편차 확인]


  train: 평균 범위 [50.0, 97.1], 40~60 이탈 유형=4개
  val: 평균 범위 [46.6, 97.0], 40~60 이탈 유형=2개
  test: 평균 범위 [49.7, 70.4], 40~60 이탈 유형=4개

[Fallback 사용 비율 — 보고서 명시]
  fb_cvr:
    train: global_fallback=2.21%, missing=0.00%
    val: global_fallback=2.48%, missing=0.00%
    test: global_fallback=1.79%, missing=0.00%
  fb_margin:
    train: global_fallback=2.21%, missing=0.00%
    val: global_fallback=2.48%, missing=0.00%
    test: global_fallback=1.79%, missing=0.00%
  fb_complete:
    train: global_fallback=2.21%, missing=0.00%
    val: global_fallback=2.48%, missing=0.00%
    test: global_fallback=1.79%, missing=0.00%
  fb_ctit:
    train: global_fallback=2.21%, missing=0.00%
    val: global_fallback=2.48%, missing=0.00%
    test: global_fallback=1.79%, missing=0.00%

OK: 점수 계산 완료


## 8. 라벨 생성 — Train 기준 Threshold (v4와 동일)

### 모델1/2 라벨 기준 차이 (v4 유지)
- 모델1: 전체 광고 기준 상위 25% = 고성과
- 모델2: 유형 x reward_band 그룹 기준 하위 25% = 부진

이 단계는 v4와 동일하다. 점수가 train fit 방식으로 바뀌었지만,
threshold는 여전히 train 점수 분포에서 quantile을 잡는다.

In [9]:
# 모델1: train 기준 상위 25% threshold 확정
THRESHOLD_TOP25 = df_train_raw['quality_score_LABEL_ONLY'].quantile(0.75)
print(f"[모델1] train 기준 고성과 threshold: {THRESHOLD_TOP25:.2f}점")

for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
    df['label_model1'] = (df['quality_score_LABEL_ONLY'] >= THRESHOLD_TOP25).astype(int)
    print(f"  {name}: 고성과 비율={df['label_model1'].mean()*100:.1f}%")

# 모델2: train 기준 그룹별 하위 25% threshold 확정
def compute_group_thresholds(df):
    group_thresholds = {}
    group_key = df['analysis_ads_type_label'].astype(str) + '_' + df['reward_band'].astype(str)

    for g in group_key.unique():
        mask = group_key == g
        if mask.sum() >= 30:
            t = df.loc[mask, 'quality_score_LABEL_ONLY'].quantile(0.25)
            group_thresholds[('group', g)] = t

    for t in df['analysis_ads_type_label'].unique():
        mask = df['analysis_ads_type_label'] == t
        if mask.sum() >= 10:
            thresh = df.loc[mask, 'quality_score_LABEL_ONLY'].quantile(0.25)
            group_thresholds[('type', t)] = thresh

    group_thresholds[('global', 'all')] = df['quality_score_LABEL_ONLY'].quantile(0.25)
    return group_thresholds

def apply_group_thresholds(df, thresholds):
    labels = pd.Series(np.nan, index=df.index)
    group_key = df['analysis_ads_type_label'].astype(str) + '_' + df['reward_band'].astype(str)

    for idx in df.index:
        g   = group_key[idx]
        t   = df.loc[idx, 'analysis_ads_type_label']
        qs  = df.loc[idx, 'quality_score_LABEL_ONLY']

        if ('group', g) in thresholds:
            labels[idx] = int(qs <= thresholds[('group', g)])
        elif ('type', t) in thresholds:
            labels[idx] = int(qs <= thresholds[('type', t)])
        else:
            labels[idx] = int(qs <= thresholds[('global', 'all')])
    return labels

print("\n[모델2] train 기준 그룹별 threshold 계산 중...")
M2_THRESHOLDS = compute_group_thresholds(df_train_raw)
print(f"그룹 threshold 수: {sum(1 for k in M2_THRESHOLDS if k[0]=='group')}개")
print(f"유형 fallback 수:  {sum(1 for k in M2_THRESHOLDS if k[0]=='type')}개")

for name, df in [('train', df_train_raw), ('val', df_val_raw), ('test', df_test_raw)]:
    df['label_model2'] = apply_group_thresholds(df, M2_THRESHOLDS)
    r = df['label_model2'].mean()
    print(f"  {name}: 부진 비율={r*100:.1f}%")

print("\n[라벨 기준 차이 — 보고서 명시 필요]")
print("  모델1: 전체 광고 기준 상위 25% (전체 고성과 탐지)")
print("  모델2: 유형 x reward_band 그룹 기준 하위 25% (유사 조건 내 부진 탐지)")

[모델1] train 기준 고성과 threshold: 65.41점
  train: 고성과 비율=25.0%
  val: 고성과 비율=21.5%
  test: 고성과 비율=25.1%

[모델2] train 기준 그룹별 threshold 계산 중...
그룹 threshold 수: 3개
유형 fallback 수:  5개


  train: 부진 비율=25.1%


  val: 부진 비율=27.0%
  test: 부진 비율=24.9%

[라벨 기준 차이 — 보고서 명시 필요]
  모델1: 전체 광고 기준 상위 25% (전체 고성과 탐지)
  모델2: 유형 x reward_band 그룹 기준 하위 25% (유사 조건 내 부진 탐지)


## 9. 초기 3일 실적 집계 + Rule-Based Flag (v4와 동일)

In [10]:
sched2 = sched.copy()
sched2['click_date_dt'] = pd.to_datetime(sched2['click_date'])
sched2['ads_sdate_dt']  = pd.to_datetime(sched2['ads_sdate'])
sched2['elapsed_day']   = (sched2['click_date_dt'] - sched2['ads_sdate_dt']).dt.days

sched2 = sched2[sched2['elapsed_day'] >= 0].copy()

early3 = (sched2[sched2['elapsed_day'] < 3]
          .groupby('ads_idx').agg(
              early_click    = ('click_cnt', 'sum'),
              early_complete = ('complete_cnt', 'sum'),
          ).reset_index())

for day in [0, 1, 2]:
    day_agg = (sched2[sched2['elapsed_day'] == day]
               .groupby('ads_idx').agg(
                   **{f'click_day{day+1}':    ('click_cnt', 'sum'),
                      f'complete_day{day+1}': ('complete_cnt', 'sum')}
               ).reset_index())
    early3 = early3.merge(day_agg, on='ads_idx', how='left')

for col in ['click_day1','click_day2','click_day3',
            'complete_day1','complete_day2','complete_day3']:
    early3[col] = early3[col].fillna(0)

early3['click_trend'] = (
    (early3['click_day3'] - early3['click_day1']) / (early3['click_day1'] + 1)
)

# 처리 방침 플래그 (v4 유지)
early3['has_early3']       = True
early3['early_click_lt10'] = (early3['early_click'] < 10).astype(int)

early3['early_cvr_raw'] = np.where(
    early3['early_click'] >= 10,
    early3['early_complete'] / early3['early_click'] * 100,
    np.nan
)

# rule_based_decline 계산
# 주의: grp_p20, grp_median은 라벨 보조 신호용이라 train 누수 영향이 작지만,
#       엄밀하게는 train 기준이 맞다. 향후 v6에서 검토 가능.
early3 = early3.merge(
    valid[['ads_idx','analysis_ads_type_label','reward_band']],
    on='ads_idx', how='inner'
)
grp_stats = (early3.dropna(subset=['early_cvr_raw'])
             .groupby(['analysis_ads_type_label','reward_band'])['early_cvr_raw']
             .agg(grp_p20=lambda x: x.quantile(0.2), grp_median='median')
             .reset_index())
early3 = early3.merge(grp_stats, on=['analysis_ads_type_label','reward_band'], how='left')

cond1 = (early3['early_cvr_raw'] < early3['grp_p20']) & (early3['early_click'] >= 10)
cond2 = early3['early_cvr_raw'] < early3['grp_median'] * 0.5
early3['rule_based_decline'] = np.where(
    early3['early_cvr_raw'].isna(), np.nan, (cond1 & cond2).astype(float)
)

print(f"초기 3일 데이터 보유: {len(early3):,}건")
print(f"  early_click >= 10 (ML 대상): {(early3['early_click']>=10).sum():,}건")
print(f"  early_click < 10  (판단보류): {(early3['early_click']<10).sum():,}건")

초기 3일 데이터 보유: 2,294건
  early_click >= 10 (ML 대상): 2,080건
  early_click < 10  (판단보류): 214건


## 10. 피처 전처리 (v4와 동일)

In [11]:
attr = (ad_attr[['ads_idx','ads_reward_price','ads_rejoin_type','ads_order',
                  'ads_action_rule','ads_action_diff_flag','ads_day_cap']]
        .drop_duplicates('ads_idx', keep='last').copy())
attr['ads_day_cap'] = attr['ads_day_cap'].astype(int)

master_feat = (ad_master[['ads_idx','regdate','ads_os_type','ads_require_adid',
                           'action_target_cnt','mentioned_media_cnt','target_media_cnt']]
               .drop_duplicates('ads_idx', keep='first').copy())
master_feat['regdate'] = pd.to_datetime(master_feat['regdate'])
master_feat['reg_hour']        = master_feat['regdate'].dt.hour
master_feat['reg_weekday_enc'] = master_feat['regdate'].dt.dayofweek
master_feat['reg_is_weekend']  = master_feat['reg_weekday_enc'].isin([5,6]).astype(int)

def hour_band(h):
    if pd.isna(h): return np.nan
    if h < 6:  return 0
    if h < 12: return 1
    if h < 18: return 2
    return 3

master_feat['reg_hour_band']    = master_feat['reg_hour'].apply(hour_band)
master_feat['ads_require_adid'] = master_feat['ads_require_adid'].astype(int)

class_feat = (ad_class[['ads_idx','final_media','final_action']]
              .drop_duplicates('ads_idx', keep='last').copy())

print("OK: 피처 전처리 완료")

OK: 피처 전처리 완료


## 11. 텍스트 원본 준비 (v4와 동일)

In [12]:
name_df    = (ad_attr[['ads_idx','ads_name']].drop_duplicates('ads_idx', keep='last').copy())
summary_df = (ad_master[['ads_idx','ads_summary']].drop_duplicates('ads_idx', keep='first').copy())
text_df = name_df.merge(summary_df, on='ads_idx', how='left')
text_df['ads_summary'] = text_df['ads_summary'].fillna('')
text_df['text_raw'] = (text_df['ads_name'].fillna('') + ' ' + text_df['ads_summary']).str.strip()

print(f"텍스트: {len(text_df):,}건 | 빈 텍스트: {(text_df['text_raw'].str.strip()=='').sum():,}건")
assert text_df['ads_idx'].nunique() == len(text_df)
print("OK: 텍스트 준비 완료")

텍스트: 445,260건 | 빈 텍스트: 0건
OK: 텍스트 준비 완료


## 12. 마트 통합

### v5 변경: fallback 컬럼을 META로 추가
점수 계산 시 어느 fallback 단계가 사용됐는지 추적 가능하도록
fb_cvr, fb_margin, fb_complete, fb_ctit를 META에 포함.

In [13]:
LABEL_ONLY_COLS = [
    'smoothed_cvr','total_margin','complete_cnt_f','avg_ctit_imputed',
    'log_margin','log_complete','log_ctit',
    'score_cvr','score_margin','score_complete','score_ctit',
    'quality_score_LABEL_ONLY',
]
META_COLS = ['is_outlier','is_outlier_cvr','is_outlier_click',
             'is_outlier_margin','is_ctit_null',
             # v5 추가: 점수 계산 fallback 추적용
             'fb_cvr','fb_margin','fb_complete','fb_ctit']

def build_mart(df_split):
    mart = df_split[[
        'ads_idx','analysis_ads_type_label','reward_band',
        *LABEL_ONLY_COLS,
        'label_model1','label_model2',
        *META_COLS,
    ]].copy()

    mart = mart.merge(
        attr[['ads_idx','ads_reward_price','ads_rejoin_type','ads_order',
              'ads_action_rule','ads_action_diff_flag','ads_day_cap']],
        on='ads_idx', how='left')

    mart = mart.merge(
        master_feat[['ads_idx','reg_hour','reg_weekday_enc','reg_is_weekend',
                     'reg_hour_band','ads_os_type','ads_require_adid',
                     'action_target_cnt','mentioned_media_cnt','target_media_cnt']],
        on='ads_idx', how='left')

    mart = mart.merge(
        class_feat[['ads_idx','final_media','final_action']],
        on='ads_idx', how='left')

    early3_cols = ['ads_idx','early_click','early_complete',
                   'click_day1','click_day2','click_day3',
                   'complete_day1','complete_day2','complete_day3',
                   'click_trend','early_cvr_raw','rule_based_decline',
                   'has_early3','early_click_lt10']
    mart = mart.merge(early3[early3_cols], on='ads_idx', how='left')

    mart['has_early3']       = mart['has_early3'].fillna(False)
    mart['early_click_lt10'] = mart['early_click_lt10'].fillna(1).astype(int)
    for col in ['early_click','early_complete','click_day1','click_day2','click_day3',
                'complete_day1','complete_day2','complete_day3','click_trend']:
        mart[col] = mart[col].fillna(0)

    mart = mart.merge(text_df[['ads_idx','text_raw']], on='ads_idx', how='left')
    mart['final_media']  = mart['final_media'].fillna('media_unknown')
    mart['final_action'] = mart['final_action'].fillna('action_unknown')
    mart['ads_os_type']  = mart['ads_os_type'].astype(str)

    assert mart['ads_idx'].nunique() == len(mart)
    return mart

mart_train = build_mart(df_train_raw)
mart_val   = build_mart(df_val_raw)
mart_test  = build_mart(df_test_raw)

print(f"mart_train: {mart_train.shape}")
print(f"mart_val:   {mart_val.shape}")
print(f"mart_test:  {mart_test.shape}")

for name, m in [('train', mart_train), ('val', mart_val), ('test', mart_test)]:
    r1 = m['label_model1'].mean()
    r2 = m['label_model2'].mean()
    print(f"  {name}: 모델1 고성과={r1*100:.1f}%, 모델2 부진={r2*100:.1f}%")
print("OK: 마트 통합 완료")

mart_train: (2176, 57)
mart_val:   (726, 57)
mart_test:  (726, 57)
  train: 모델1 고성과=25.0%, 모델2 부진=25.1%
  val: 모델1 고성과=21.5%, 모델2 부진=27.0%
  test: 모델1 고성과=25.1%, 모델2 부진=24.9%
OK: 마트 통합 완료


## 13. OHE + TF-IDF + 피처 행렬 생성 (v4와 동일)

### v5 누수 체크 추가
fb_* (fallback level) 컬럼은 LABEL_ONLY/META 영역. 피처에 들어가면 안 됨.

In [14]:
NUMERIC_FEATS = [
    'ads_reward_price','ads_rejoin_type','ads_order',
    'ads_action_rule','ads_action_diff_flag','ads_day_cap',
    'reg_hour','reg_hour_band','reg_weekday_enc','reg_is_weekend',
    'ads_require_adid','action_target_cnt',
    'mentioned_media_cnt','target_media_cnt',
]
OHE_FEATS = [
    'analysis_ads_type_label','reward_band',
    'final_media','final_action','ads_os_type',
]
EARLY_FEATS = [
    'early_click','early_complete',
    'click_day1','click_day2','click_day3',
    'complete_day1','complete_day2','complete_day3',
    'click_trend',
]

# 누수 체크 (v5: fb_* 도 추가)
LABEL_ONLY_ALL = LABEL_ONLY_COLS + ['early_cvr_raw']
FORBIDDEN = set(LABEL_ONLY_ALL + META_COLS +
                ['rule_based_decline','has_early3','early_click_lt10'])
for col in NUMERIC_FEATS + OHE_FEATS + EARLY_FEATS:
    assert col not in FORBIDDEN, f"누수 위험: {col}"
assert 'is_outlier' not in NUMERIC_FEATS + OHE_FEATS + EARLY_FEATS
for fb in ['fb_cvr','fb_margin','fb_complete','fb_ctit']:
    assert fb not in NUMERIC_FEATS + OHE_FEATS + EARLY_FEATS, f"fallback 컬럼 누수: {fb}"
print("OK: 누수 체크 통과 (fb_* 포함)")

# OHE fit (train에서만)
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(mart_train[OHE_FEATS].astype(str))
ohe_feature_names = ohe.get_feature_names_out(OHE_FEATS).tolist()
print(f"OHE 피처: {len(ohe_feature_names)}개")

# TF-IDF fit (train에서만)
TFIDF_PARAMS = dict(analyzer='char_wb', ngram_range=(2,4),
                     max_features=50, min_df=2, sublinear_tf=True)
tfidf = TfidfVectorizer(**TFIDF_PARAMS)
tfidf.fit(mart_train['text_raw'].fillna('').tolist())
tfidf_feature_names = [f"tfidf_{n}" for n in tfidf.get_feature_names_out()]
print(f"TF-IDF 피처: {len(tfidf_feature_names)}개")

def build_X(mart, include_early=False):
    ohe_arr   = ohe.transform(mart[OHE_FEATS].astype(str))
    tfidf_arr = tfidf.transform(mart['text_raw'].fillna('').tolist()).toarray()
    parts = [
        mart[NUMERIC_FEATS].reset_index(drop=True),
        pd.DataFrame(ohe_arr,   columns=ohe_feature_names),
        pd.DataFrame(tfidf_arr, columns=tfidf_feature_names),
    ]
    if include_early:
        parts.append(mart[EARLY_FEATS].reset_index(drop=True))
    return pd.concat(parts, axis=1)

X_train_m1 = build_X(mart_train, include_early=False)
X_val_m1   = build_X(mart_val,   include_early=False)
X_test_m1  = build_X(mart_test,  include_early=False)

X_train_m2 = build_X(mart_train, include_early=True)
X_val_m2   = build_X(mart_val,   include_early=True)
X_test_m2  = build_X(mart_test,  include_early=True)

print(f"\n[피처 수 — 순수 피처만]")
print(f"  모델1: {len(NUMERIC_FEATS)}(수치) + {len(ohe_feature_names)}(OHE) + {len(tfidf_feature_names)}(TF-IDF) = {X_train_m1.shape[1]}개")
print(f"  모델2: {X_train_m1.shape[1]}(모델1) + {len(EARLY_FEATS)}(초기3일) = {X_train_m2.shape[1]}개")
print(f"\n[저장 파일 컬럼 수]")
print(f"  모델1 저장: {X_train_m1.shape[1] + 3}개 (피처 + label + is_outlier + ads_idx)")
print(f"  모델2 저장: {X_train_m2.shape[1] + 3}개")

OK: 누수 체크 통과 (fb_* 포함)
OHE 피처: 34개


TF-IDF 피처: 50개



[피처 수 — 순수 피처만]
  모델1: 14(수치) + 34(OHE) + 50(TF-IDF) = 98개
  모델2: 98(모델1) + 9(초기3일) = 107개

[저장 파일 컬럼 수]
  모델1 저장: 101개 (피처 + label + is_outlier + ads_idx)
  모델2 저장: 110개


## 14. 저장 + 전처리 객체 저장

### v5 추가 저장 항목
- `scorer`: GroupedECDFScorer 객체 (production 재현용)
- `score_value_cols`, `score_col_map`, `fallback_col_map`: 매핑 정보
- `stratify_key_col`: 사용한 stratify 키 ('log_margin' quintile)

In [15]:
def save_split(X, y, outlier, ads_idx, path):
    df_out = X.copy()
    df_out.index = range(len(df_out))
    df_out['label']      = y.values
    df_out['is_outlier'] = outlier.values
    df_out['ads_idx']    = ads_idx.values
    df_out.to_parquet(path, index=False)
    fname = path.split('\\')[-1] if '\\' in path else path.split('/')[-1]
    n_feat = X.shape[1]
    print(f"저장: {fname} | 피처={n_feat}개 | 저장컬럼={df_out.shape[1]}개 | label={y.mean()*100:.1f}%")

# 모델1
save_split(X_train_m1, mart_train['label_model1'], mart_train['is_outlier'],
           mart_train['ads_idx'], OUT + 'model1_train.parquet')
save_split(X_val_m1,   mart_val['label_model1'],   mart_val['is_outlier'],
           mart_val['ads_idx'],   OUT + 'model1_val.parquet')
save_split(X_test_m1,  mart_test['label_model1'],  mart_test['is_outlier'],
           mart_test['ads_idx'],  OUT + 'model1_test.parquet')

# 모델2
save_split(X_train_m2, mart_train['label_model2'], mart_train['is_outlier'],
           mart_train['ads_idx'], OUT + 'model2_train.parquet')
save_split(X_val_m2,   mart_val['label_model2'],   mart_val['is_outlier'],
           mart_val['ads_idx'],   OUT + 'model2_val.parquet')
save_split(X_test_m2,  mart_test['label_model2'],  mart_test['is_outlier'],
           mart_test['ads_idx'],  OUT + 'model2_test.parquet')

# 마트 원본
mart_all = pd.concat([mart_train, mart_val, mart_test], ignore_index=True)
mart_all.to_parquet(OUT + 'ad_mart_v5.parquet', index=False)
print(f"\n마트 원본: ad_mart_v5.parquet | shape={mart_all.shape}")

# 전처리 객체 저장 (v5: scorer 추가)
pipeline_artifacts = {
    'ohe':               ohe,
    'tfidf':             tfidf,
    'ohe_feature_names': ohe_feature_names,
    'tfidf_feature_names': tfidf_feature_names,
    'numeric_feats':     NUMERIC_FEATS,
    'ohe_feats':         OHE_FEATS,
    'early_feats':       EARLY_FEATS,
    'model1_feature_cols': list(X_train_m1.columns),
    'model2_feature_cols': list(X_train_m2.columns),
    'label_model1_threshold': float(THRESHOLD_TOP25),
    'label_model2_group_thresholds': M2_THRESHOLDS,
    'global_cvr':        float(GLOBAL_CVR),
    'ctit_median':       float(ctit_median),
    # v5 추가
    'scorer':            scorer,
    'score_value_cols':  SCORE_VALUE_COLS,
    'score_col_map':     SCORE_COL_MAP,
    'fallback_col_map':  FALLBACK_COL_MAP,
    'stratify_key':      'log_margin_quintile',
    'version':           'v5',
}
joblib.dump(pipeline_artifacts, OUT + 'preprocessing_pipeline.joblib')
print("\n전처리 객체 저장: preprocessing_pipeline.joblib")
print("  v5 추가: scorer (GroupedECDFScorer), 점수 매핑, stratify_key")

저장: model1_train.parquet | 피처=98개 | 저장컬럼=101개 | label=25.0%
저장: model1_val.parquet | 피처=98개 | 저장컬럼=101개 | label=21.5%
저장: model1_test.parquet | 피처=98개 | 저장컬럼=101개 | label=25.1%
저장: model2_train.parquet | 피처=107개 | 저장컬럼=110개 | label=25.1%
저장: model2_val.parquet | 피처=107개 | 저장컬럼=110개 | label=27.0%
저장: model2_test.parquet | 피처=107개 | 저장컬럼=110개 | label=24.9%

마트 원본: ad_mart_v5.parquet | shape=(3628, 57)

전처리 객체 저장: preprocessing_pipeline.joblib
  v5 추가: scorer (GroupedECDFScorer), 점수 매핑, stratify_key


## 15. 최종 검증 체크리스트

In [16]:
print("=" * 65)
print("최종 검증 체크리스트 (v5)")
print("=" * 65)

LABEL_ONLY_ALL = LABEL_ONLY_COLS + ['early_cvr_raw']
META_ALL = META_COLS + ['rule_based_decline','has_early3','early_click_lt10']

checks = [
    ("Split 먼저 -> train 기준 threshold 적용 (v4 유지)",
     True),

    ("[v5] 품질 점수: train fit ECDF 방식으로 변경",
     hasattr(scorer, 'fitted_') and scorer.fitted_),

    ("[v5] ctit_median: train 기준",
     abs(ctit_median - df_train_raw['avg_ctit'].median()) < 1e-9),

    ("[v5] GLOBAL_CVR: train 기준",
     abs(GLOBAL_CVR - df_train_raw['complete_cnt'].sum() /
                      df_train_raw['click_cnt'].sum()) < 1e-9),

    ("[v5] Stratify 키: log_margin quintile (raw 변수 기반)",
     '_strat' in valid.columns),

    ("모델1 피처에 LABEL_ONLY 미포함",
     all(c not in X_train_m1.columns for c in LABEL_ONLY_ALL)),

    ("모델1 피처에 is_outlier 미포함",
     'is_outlier' not in X_train_m1.columns),

    ("[v5] 모델1/2 피처에 fb_* (fallback level) 미포함",
     all(fb not in X_train_m1.columns and fb not in X_train_m2.columns
         for fb in ['fb_cvr','fb_margin','fb_complete','fb_ctit'])),

    ("모델2 피처에 early_cvr_raw 미포함",
     'early_cvr_raw' not in X_train_m2.columns),

    ("모델2 피처에 최종 성과 미포함",
     all(c not in X_train_m2.columns for c in
         ['cvr_pct','smoothed_cvr','total_margin','quality_score_LABEL_ONLY'])),

    ("OHE train only fit", True),
    ("TF-IDF train only fit", True),

    ("모델2 has_early3 / early_click_lt10 flag 추가",
     'has_early3' in mart_train.columns and 'early_click_lt10' in mart_train.columns),

    ("전처리 객체 joblib 저장 완료",
     os.path.exists(OUT + 'preprocessing_pipeline.joblib')),

    ("is_valid_click10 필터 적용 대상 명확화",
     True),
]

all_pass = True
for desc, result in checks:
    status = "OK" if result else "FAIL"
    print(f"  [{status}] {desc}")
    if not result:
        all_pass = False

print()
print(f"[피처 수 최종 확인]")
print(f"  모델1 순수 피처: {X_train_m1.shape[1]}개")
print(f"  모델2 순수 피처: {X_train_m2.shape[1]}개")
print(f"  모델1 저장 컬럼: {X_train_m1.shape[1]+3}개 (피처+label+is_outlier+ads_idx)")
print(f"  모델2 저장 컬럼: {X_train_m2.shape[1]+3}개")
print()
print(f"[Fallback 사용 비율 — 보고서 명시]")
for fb_col in ['fb_cvr', 'fb_margin', 'fb_complete', 'fb_ctit']:
    for name, m in [('val', mart_val), ('test', mart_test)]:
        lvl = m[fb_col]
        pct_g = (lvl == 1).mean() * 100
        pct_m = (lvl == 2).mean() * 100
        if pct_g > 0 or pct_m > 0:
            print(f"  {fb_col} {name}: global_fb={pct_g:.2f}%, missing={pct_m:.2f}%")
print()
print(f"[모델 적용 대상 명확화]")
print(f"  모델1/2: is_valid_click10==1 (클릭>=10) 광고만 대상")
print(f"  모델2:   추가로 early_click>=10인 경우만 ML 예측 (나머지 판단보류)")
print()
print(f"[보고서 필수 명시 사항]")
print(f"  1. 모델 적용 대상: 클릭>=10 확보 광고 (표본 선택 편향 가능성)")
print(f"  2. 모델1 라벨: 전체 기준 상위 25%")
print(f"  3. 모델2 라벨: 유형x보상대 그룹 기준 하위 25%")
print(f"  4. Split: Stratified Random (시간기반 불가 — 92%가 25년 7~8월 집중)")
print(f"  5. [v5] 품질 점수: train ECDF fit -> val/test transform (target leakage 차단)")
print(f"  6. [v5] Fallback 사용 비율은 위 출력 참고")
print()
if all_pass:
    print("전체 통과 — ML 마트 v5 생성 완료")
else:
    print("일부 실패 — 위 내용 확인 필요")

최종 검증 체크리스트 (v5)
  [OK] Split 먼저 -> train 기준 threshold 적용 (v4 유지)
  [OK] [v5] 품질 점수: train fit ECDF 방식으로 변경
  [OK] [v5] ctit_median: train 기준
  [OK] [v5] GLOBAL_CVR: train 기준
  [OK] [v5] Stratify 키: log_margin quintile (raw 변수 기반)
  [OK] 모델1 피처에 LABEL_ONLY 미포함
  [OK] 모델1 피처에 is_outlier 미포함
  [OK] [v5] 모델1/2 피처에 fb_* (fallback level) 미포함
  [OK] 모델2 피처에 early_cvr_raw 미포함
  [OK] 모델2 피처에 최종 성과 미포함
  [OK] OHE train only fit
  [OK] TF-IDF train only fit
  [OK] 모델2 has_early3 / early_click_lt10 flag 추가
  [OK] 전처리 객체 joblib 저장 완료
  [OK] is_valid_click10 필터 적용 대상 명확화

[피처 수 최종 확인]
  모델1 순수 피처: 98개
  모델2 순수 피처: 107개
  모델1 저장 컬럼: 101개 (피처+label+is_outlier+ads_idx)
  모델2 저장 컬럼: 110개

[Fallback 사용 비율 — 보고서 명시]
  fb_cvr val: global_fb=2.48%, missing=0.00%
  fb_cvr test: global_fb=1.79%, missing=0.00%
  fb_margin val: global_fb=2.48%, missing=0.00%
  fb_margin test: global_fb=1.79%, missing=0.00%
  fb_complete val: global_fb=2.48%, missing=0.00%
  fb_complete test: global_fb=1.79%, missing=0.00%
  fb